# Overnight WTI Baseline Runner

This notebook runs the full overnight WTI baseline batch and saves outputs to Google Drive or local Colab storage.

Batch included by default:
- univariate daily
- univariate weekly
- multivariate daily
- multivariate weekly

Models included by default:
- GRU
- TimeXer
- iTransformer

Each run saves:
- train / validation loss history CSV
- loss curve PNG
- forecast plot PNG
- metrics CSV
- predictions CSV
- per-run config snapshot
- batch summary CSV / HTML / Markdown report


In [ ]:
%pip -q install "git+https://github.com/Nixtla/neuralforecast.git" pandas matplotlib openpyxl pyyaml


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jaeho777/newoil.git"
WORKDIR = Path("/content/newoil")
BATCH_CONFIG_RELATIVE_PATH = "configs/batches/overnight_wti_baseline.yaml"  # Use configs/batches/smoke_wti_baseline.yaml first if you want a quick safety check.

SAVE_TO_GOOGLE_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/newoil_outputs")
LOCAL_OUTPUT_ROOT = Path("/content/newoil_outputs")

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORKDIR)], check=True)
sys.path.insert(0, str(WORKDIR / "src"))

from newoil import run_batch_from_config

repo_root = WORKDIR
batch_config_path = repo_root / BATCH_CONFIG_RELATIVE_PATH
output_root = DRIVE_OUTPUT_ROOT if SAVE_TO_GOOGLE_DRIVE else LOCAL_OUTPUT_ROOT
output_root.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {repo_root}")
print(f"Batch config: {batch_config_path}")
print(f"Output root: {output_root}")


In [ ]:
result = run_batch_from_config(
    batch_config_path=batch_config_path,
    repo_root=repo_root,
    output_root=output_root,
)

summary_df = result.summary_df.copy()
summary_df


In [ ]:
from IPython.display import Image, Markdown, display
import pandas as pd

display(Markdown(f"# Overnight Report\n\n- Batch dir: `{result.batch_dir}`\n- Summary CSV: `{result.batch_dir / 'summary.csv'}`\n- HTML report: `{result.report_html}`"))
display(summary_df)

for _, row in summary_df.iterrows():
    display(Markdown(f"## {row['run_name']}"))
    if row['status'] == 'failed':
        display(Markdown(f"Failed: `{row.get('error_message', '')}`"))
        continue

    artifact_dir = Path(row['artifact_dir'])
    metrics_df = pd.read_csv(artifact_dir / 'metrics.csv')
    display(metrics_df)
    display(Image(filename=str(artifact_dir / 'loss_curve.png')))
    display(Image(filename=str(artifact_dir / 'forecast_plot.png')))
